# Gold - ecommerce_pedidos

Notebook para criação de tabelas Gold em Delta Lake e réplica opcional para SQL Server/Azure, mantendo padrão de consumo analítico via Looker.

Este notebook assume que as tabelas Silver e `squad1.dq_monitoring_logs` já foram criadas em Delta.

In [0]:
%run ../utils/utils

## Inicialização

In [0]:

import uuid
import pyspark.sql.functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone

print("Iniciando processamento da Camada Gold - Data Quality")

## VARIÁVEL DE CONTROLE

In [0]:


# ---------------------------------------------------------
# VARIÁVEL DE CONTROLE: Definição obrigatória
# ---------------------------------------------------------
TABELA_ALVO = "ecommerce_rastreamento"

# Nomes dinâmicos das tabelas Gold geradas
tabela_gold_regras = f"gold_dq_regras_{TABELA_ALVO}"
tabela_gold_saude = f"gold_dq_saude_{TABELA_ALVO}"
tabela_gold_saude_percentual = f"gold_saude_percentual_{TABELA_ALVO}"
tabela_quarentena = f"gold_dq_quarentena_{TABELA_ALVO}"

# Coluna de ID natural de cada tabela — usada para contar distintos na Bronze
# na hora de calcular o percentual de saúde. Mantido consistente com o
# insights.py e o notebook de volumetria.
IDS_POR_TABELA = {
    "ecommerce_clientes": "id_cliente",
    "ecommerce_pedidos": "id_pedido",
    "ecommerce_enderecos": "id_endereco",
    "ecommerce_itens_pedido": "id_item_pedido",
    "ecommerce_rastreamento": "id_rastreamento",
    "ecommerce_categorias": "id_categoria",
}

print(f"===== INICIANDO PROCESSAMENTO DA CAMADA GOLD PARA: {TABELA_ALVO} =====")


## PROCESSAMENTO DAS REGRAS

In [0]:
# ==============================================================================
# 1. PROCESSAMENTO DAS REGRAS (Dashboard 1: Falhas por Dia e Regra)
# ==============================================================================
print(f"Processando Resumo por Regra para {TABELA_ALVO}...")

if delta_existe(camada="", tabela="dq_monitoring_logs", storage_opts=STORAGE_OPTIONS):
    # Lê os logs gerais e filtra EXCLUSIVAMENTE a tabela alvo
    df_logs = ler_delta(camada="", tabela="dq_monitoring_logs", storage_opts=STORAGE_OPTIONS) \
        .filter(F.col("tabela") == TABELA_ALVO)

    if df_logs.count() > 0:
        df_gold_regras = df_logs \
            .withColumn("data_execucao", F.to_date("timestamp_execucao")) \
            .groupBy("data_execucao", "tabela", "regra", "severidade") \
            .agg(
                F.sum("qtd_registros_falhos").alias("total_falhas"),
                F.sum("qtd_registros_total").alias("total_processado")
            ) \
            .withColumn(
                "perc_falha",
                F.round((F.col("total_falhas") / F.col("total_processado")) * 100, 2)
            ) \
            .orderBy(F.col("data_execucao").desc(), F.col("perc_falha").desc())

        print(f"-> Resumo por Regra gerado com sucesso ({df_gold_regras.count()} linhas).")
    else:
        print(f"-> Aviso: Nenhum log de falha encontrado para {TABELA_ALVO}.")
        df_gold_regras = None
else:
    print("-> Aviso: Tabela dq_monitoring_logs não encontrada no Data Lake.")
    df_gold_regras = None

##  PROCESSAMENTO DE SAÚDE POR TABELA

In [0]:
# ==============================================================================
# 2. PROCESSAMENTO DE SAÚDE POR TABELA (Volume de registros entregues por Hora)
# ==============================================================================
print(f"\nProcessando Saúde por Hora para {TABELA_ALVO}...")

if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)

    df_gold_saude = df_silver \
        .withColumn("data_hora", F.date_trunc("hour", F.col("silver_processed_at"))) \
        .groupBy("data_hora") \
        .agg(
            # Como a Silver só armazena dados que passaram na validação,
            # o total de linhas É o total de registros limpos.
            F.count("*").alias("qtd_limpos")
        ) \
        .withColumn("tabela", F.lit(TABELA_ALVO)) \
        .withColumn("qtd_total", F.col("qtd_limpos")) \
        .withColumn("perc_limpos", F.lit(100.0)) \
        .select("data_hora", "tabela", "qtd_total", "qtd_limpos", "perc_limpos") \
        .orderBy(F.col("data_hora").desc())

    print(f"-> Resumo de Saúde gerado com sucesso ({df_gold_saude.count()} horas processadas).")
else:
    print(f"-> Aviso: Tabela Silver {TABELA_ALVO} não encontrada.")
    df_gold_saude = None


In [0]:
print(f"\nProcessando Saúde da Tabela (% invalidado) para {TABELA_ALVO}...")

id_col = IDS_POR_TABELA.get(TABELA_ALVO)

qtd_bronze_distintos = 0
qtd_quarentena_saude = 0
qtd_silver_saude = 0
percentual_invalidado = None
percentual_saude = None

if id_col is None:
    print(f"-> Aviso: coluna de ID não mapeada para {TABELA_ALVO} em IDS_POR_TABELA. "
          "Saúde da tabela não pôde ser calculada.")
    df_gold_saude_percentual = None
else:
    # 1. Obtém os IDs distintos da Bronze ATUAL (snapshot de hoje)
    # CORREÇÃO: mesmo reconstruindo via spark.createDataFrame(pandas_df), o
    # Spark Connect (usado no Serverless) ainda pode acionar algo
    # equivalente a PERSIST TABLE internamente. Para eliminar essa
    # dependência de vez, a interseção agora é feita INTEIRAMENTE em Python
    # puro (set), sem nenhuma operação Spark no meio — só toPandas() no
    # início pra trazer os IDs pra memória local.
    set_ids_bronze_atual = set()
    if delta_existe("bronze", TABELA_ALVO, STORAGE_OPTIONS):
        pdf_ids_bronze_atual = (
            ler_delta("bronze", TABELA_ALVO, STORAGE_OPTIONS)
            .select(id_col).where(F.col(id_col).isNotNull()).distinct()
            .toPandas()
        )
        set_ids_bronze_atual = set(pdf_ids_bronze_atual[id_col])
        qtd_bronze_distintos = len(set_ids_bronze_atual)
    else:
        print(f"-> Aviso: tabela Bronze de {TABELA_ALVO} não encontrada.")

    # 2. Obtém o total da Quarentena, intersectado (em Python) com o snapshot da Bronze
    if id_col is not None and set_ids_bronze_atual and delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
        pdf_quarentena_ids = (
            ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
            .select(id_col).where(F.col(id_col).isNotNull()).distinct()
            .toPandas()
        )
        set_quarentena_ids = set(pdf_quarentena_ids[id_col])
        qtd_quarentena_saude = len(set_ids_bronze_atual & set_quarentena_ids)

    # Trava defensiva também nos números BRUTOS (não só no percentual) —
    # garante matematicamente que o numerador nunca excede o denominador.
    if qtd_bronze_distintos > 0:
        qtd_quarentena_saude = min(qtd_quarentena_saude, qtd_bronze_distintos)

    # 3. CHECAGEM DA SILVER: Obtém o total de registros válidos na Silver
    silver_valida_e_povoada = False
    if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
        qtd_silver_saude = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS).count()
        if qtd_silver_saude > 0:
            silver_valida_e_povoada = True

    # 4. Cálculo dos Percentuais
    # CORREÇÃO: o clamp agora é INCONDICIONAL e sempre roda por último, antes
    # de calcular percentual_saude — assim os dois valores SEMPRE somam 100,
    # e é matematicamente impossível gerar um "acima de 100%" de um lado e
    # "negativo" do outro, não importa qual ramo (populada ou não) gerou o
    # número bruto.
    if qtd_bronze_distintos > 0:
        if not silver_valida_e_povoada:
            percentual_bruto = 100.0
            qtd_quarentena_saude = qtd_bronze_distintos
        else:
            percentual_bruto = (qtd_quarentena_saude / qtd_bronze_distintos) * 100

        if percentual_bruto > 100 or percentual_bruto < 0:
            print(f"-> Aviso: percentual bruto calculado ({round(percentual_bruto, 2)}%) fora do "
                  "intervalo [0, 100] — há inconsistência entre Bronze e Quarentena. "
                  "Valor foi limitado (clamp), mas vale investigar a causa.")

        percentual_invalidado = round(max(0.0, min(100.0, percentual_bruto)), 2)
        percentual_saude = round(100.0 - percentual_invalidado, 2)

    # 5. Geração do DataFrame da Gold de Saúde
    if percentual_invalidado is not None:
        df_gold_saude_percentual = spark.createDataFrame(
            [(TABELA_ALVO, qtd_bronze_distintos, qtd_quarentena_saude, percentual_invalidado, percentual_saude)],
            ["tabela", "total_bronze_distintos", "total_quarentena", "percentual_invalidado", "percentual_saude"]
        ).withColumn("data_verificacao", F.current_timestamp())

        print(
            f"-> Saúde da tabela: {percentual_saude}% passou para a Silver, "
            f"{percentual_invalidado}% foi invalidado ({qtd_quarentena_saude} de {qtd_bronze_distintos})."
        )
    else:
        print("-> Aviso: não foi possível calcular o percentual (Bronze vazia).")
        df_gold_saude_percentual = None

## SALVAMENTO 1: DELTA LAKE

In [0]:
# ==============================================================================
# 3. SALVAMENTO 1: DELTA LAKE (A Fonte da Verdade na Camada Gold)
# ==============================================================================
print("\nIniciando salvamento no DELTA LAKE...")

if df_gold_regras is not None:
    gravar_delta(
        df=df_gold_regras,
        camada="gold",
        tabela=tabela_gold_regras,
        storage_opts=STORAGE_OPTIONS,
        mode="overwrite",
        particionar=False
    )
    print(f"-> Sucesso: {tabela_gold_regras} gravada em delta (pasta gold).")

if df_gold_saude is not None:
    gravar_delta(
        df=df_gold_saude,
        camada="gold",
        tabela=tabela_gold_saude,
        storage_opts=STORAGE_OPTIONS,
        mode="overwrite",
        particionar=False
    )
    print(f"-> Sucesso: {tabela_gold_saude} gravada em delta (pasta gold).")

if df_gold_saude_percentual is not None:
    gravar_delta(
        df=df_gold_saude_percentual,
        camada="gold",
        tabela=tabela_gold_saude_percentual,
        storage_opts=STORAGE_OPTIONS,
        mode="overwrite",
        particionar=False
    )
    print(f"-> Sucesso: {tabela_gold_saude_percentual} gravada em delta (pasta gold).")

## SALVAMENTO 2: AZURE SQL SERVER

In [0]:
# ==============================================================================
# 4. SALVAMENTO 2: AZURE SQL SERVER (Camada de Serviço para o Looker)
# ==============================================================================
print("\nIniciando espelhamento no AZURE SQL SERVER...")

def salvar_sql_server(df, nome_tabela):
    try:
        # Passamos a porta padrão 1433 diretamente na configuração
        df.write \
            .format("sqlserver") \
            .option("host", JDBC_HOSTNAME) \
            .option("port", "1433") \
            .option("database", JDBC_DATABASE) \
            .option("dbtable", f"squad1.{nome_tabela}") \
            .option("user", JDBC_USERNAME) \
            .option("password", JDBC_PASSWORD) \
            .option("trustServerCertificate", "true") \
            .mode("overwrite") \
            .save()

        print(f"-> Sucesso: Tabela squad1.{nome_tabela} atualizada no SQL Server.")
    except Exception as e:
        print(f"-> Erro ao enviar {nome_tabela} ao SQL Server: {e}")

if df_gold_regras is not None:
    salvar_sql_server(df_gold_regras, tabela_gold_regras)

if df_gold_saude is not None:
    salvar_sql_server(df_gold_saude, tabela_gold_saude)

if df_gold_saude_percentual is not None:
    salvar_sql_server(df_gold_saude_percentual, tabela_gold_saude_percentual)

print(f"\n===== CAMADA GOLD FINALIZADA PARA {TABELA_ALVO} =====")


%md


## EXPORTAÇÃO DA QUARENTENA PARA O BANCO DE DADOS


In [0]:
# ==============================================================================
# EXPORTAÇÃO DA QUARENTENA DA SILVER PARA O BANCO DE DADOS (POSTGRESQL)
# ==============================================================================

print(f"Iniciando leitura da Quarentena de {TABELA_ALVO} no Data Lake...")

# 1. Verifica se a tabela de quarentena existe na subpasta da silver no Data Lake
if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):

    # 2. Carrega os dados físicos da quarentena da Silver
    df_quarentena_silver = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
    qtd_rejeitados = df_quarentena_silver.count()

    print(f"Registros encontrados na quarentena de {TABELA_ALVO}: {qtd_rejeitados}")

    if qtd_rejeitados > 0:

        # 3. Tratamento de NullType para garantir compatibilidade estrita com o Postgres
        colunas_corrigidas = [
            F.col(campo.name).cast("string") if str(campo.dataType).lower().startswith("null") else F.col(campo.name)
            for campo in df_quarentena_silver.schema.fields
        ]
        df_quarentena_pronta = df_quarentena_silver.select(*colunas_corrigidas)

        print("Enviando dados para o PostgreSQL...")

        # 4. Gravação física usando o conector nativo "postgresql" (Removido ssl/sslmode para conformidade Serverless)
        df_quarentena_pronta.write \
            .format("sqlserver") \
            .option("host", JDBC_HOSTNAME) \
            .option("port", "1433") \
            .option("database", JDBC_DATABASE) \
            .option("dbtable", f"squad1.{tabela_quarentena}") \
            .option("user", JDBC_USERNAME) \
            .option("password", JDBC_PASSWORD) \
            .option("trustServerCertificate", "true") \
            .mode("overwrite") \
            .save()

        print(f"[Sucesso] {qtd_rejeitados} registros da Quarentena foram gravados na tabela squad1.{tabela_quarentena} no PostgreSQL!")
    else:
        print("Quarentena vazia no Data Lake. Nenhum dado pendente para enviar nesta execução.")
else:
    print(f"A tabela de quarentena para {TABELA_ALVO} ainda não existe no Data Lake.")
